# Custom Evaluation with the Galileo Python SDK

This notebook keeps the Galileo SDK calls inline so you can see how to register metrics, log traces, and score custom workflows without relying on the wrapper class.


In [1]:
import sys
from pathlib import Path

from dotenv import load_dotenv

for root in [Path.cwd(), Path.cwd().parent]:
    if str(root) not in sys.path:
        sys.path.insert(0, str(root))

env_candidates = [Path.cwd() / '.env', Path.cwd().parent / '.env']
for candidate in env_candidates:
    if candidate.exists():
        load_dotenv(candidate)
        ENV_FILE = candidate
        break
else:
    raise FileNotFoundError('Could not find .env in the repo root or current directory')

print(f'Loaded credentials from {ENV_FILE}')


Loaded credentials from /Users/binliu/Documents/rungalileo/galileo-test/.env


In [ ]:
from galileo import GalileoLogger, GalileoMetrics, Message, MessageRole, galileo_context
from galileo.config import GalileoPythonConfig
from galileo.decorator import log as galileo_log
from galileo.log_streams import LocalMetricConfig, create_log_stream, enable_metrics, get_log_stream
from galileo.projects import delete_project
from galileo.scorers import Scorers

PROJECT_NAME = 'GalileoEval_S6_CustomEval'
LOG_STREAM_NAME = 'custom-eval-pipeline'
MODEL = 'gpt-4o-mini'

## 1. Initialize Galileo

Initialize the Galileo context, make sure the log stream exists, and print the console links if Galileo returns IDs.


In [ ]:
galileo_context.init(project=PROJECT_NAME, log_stream=LOG_STREAM_NAME)

log_stream = get_log_stream(name=LOG_STREAM_NAME, project_name=PROJECT_NAME)
if not log_stream:
    log_stream = create_log_stream(name=LOG_STREAM_NAME, project_name=PROJECT_NAME)

config = GalileoPythonConfig.get()
logger = galileo_context.get_logger_instance()
project_id = getattr(logger, 'project_id', None)
log_stream_id = getattr(logger, 'log_stream_id', None)

if project_id and log_stream_id:
    project_url = f'{config.console_url}project/{project_id}'
    log_stream_url = f'{project_url}/log-streams/{log_stream_id}'
    print(project_url)
    print(log_stream_url)
else:
    print('Galileo context initialized')

## 2. Register a Local Metric


In [5]:
def response_length_scorer(trace, **kwargs) -> float:
    output = str(trace.output or '')
    length = len(output)
    if 20 <= length <= 200:
        return 1.0
    if length < 20:
        return length / 20.0
    return max(0.0, 1.0 - (length - 200) / 500.0)

enable_metrics(
    project_name=PROJECT_NAME,
    log_stream_name=LOG_STREAM_NAME,
    metrics=[LocalMetricConfig(name='response_length_quality', scorer_fn=response_length_scorer)],
)


[LocalMetricConfig(name='response_length_quality', scorer_fn=<function response_length_scorer at 0x1100c9b20>, aggregator_fn=None, scorable_types=[<StepType.llm: 'llm'>], aggregatable_types=[<StepType.trace: 'trace'>])]

## 3. Log a Decorated Workflow


In [6]:
galileo_context.init(project=PROJECT_NAME, log_stream=LOG_STREAM_NAME)

@galileo_log(span_type='retriever')
def fetch_docs(query: str) -> list[str]:
    return [f'Document about: {query}', f'Technical reference for: {query}']

@galileo_log(span_type='tool')
def format_response(docs: list[str], query: str) -> str:
    return f"Based on {len(docs)} documents for {query}: {', '.join(docs)}"

@galileo_log(span_type='workflow')
def qa_pipeline(question: str) -> str:
    docs = fetch_docs(question)
    return format_response(docs, question)

result = qa_pipeline('How does custom evaluation work in Galileo?')
galileo_context.flush()
result


'Based on 2 documents for How does custom evaluation work in Galileo?: Document about: How does custom evaluation work in Galileo?, Technical reference for: How does custom evaluation work in Galileo?'

## 4. Log Manual Traces for Evaluation


In [7]:
logger = GalileoLogger(project=PROJECT_NAME, log_stream=LOG_STREAM_NAME)

logger.start_trace(input='What is the capital of France?')
logger.add_llm_span(
    input=[Message(role=MessageRole.user, content='What is the capital of France?')],
    output=Message(role=MessageRole.assistant, content='The capital of France is Paris.'),
    model=MODEL,
    num_input_tokens=8,
    num_output_tokens=7,
)
logger.conclude(output='The capital of France is Paris.')

logger.start_trace(input='Explain quantum computing.')
logger.add_llm_span(
    input=[Message(role=MessageRole.user, content='Explain quantum computing.')],
    output=Message(role=MessageRole.assistant, content="It's complex."),
    model=MODEL,
    num_input_tokens=5,
    num_output_tokens=2,
)
logger.conclude(output="It's complex.")

logger.start_trace(input='What is machine learning?')
logger.add_llm_span(
    input=[Message(role=MessageRole.user, content='What is machine learning?')],
    output=Message(
        role=MessageRole.assistant,
        content='Machine learning is a subset of artificial intelligence that enables systems to learn and improve from experience without being explicitly programmed.',
    ),
    model=MODEL,
    num_input_tokens=6,
    num_output_tokens=25,
)
logger.conclude(output='Machine learning is a subset of AI that enables systems to learn from experience.')
logger.flush()
'Logged 3 traces'


'Logged 3 traces'

## 5. Enable Safety Metrics


In [8]:
enable_metrics(
    project_name=PROJECT_NAME,
    log_stream_name=LOG_STREAM_NAME,
    metrics=[
        GalileoMetrics.prompt_injection,
        GalileoMetrics.input_sexist,
        GalileoMetrics.output_sexist,
        GalileoMetrics.input_toxicity,
        GalileoMetrics.output_toxicity,
    ],
)


[]

## 6. Inspect Available Scorers


In [9]:
len(Scorers().list())


141

## 7. Get Tracing Headers for Distributed Tracing

`get_tracing_headers()` is only available when the logger is created in **distributed** mode. This mode is used when one service starts a trace and passes trace context to a downstream service so both services contribute spans to the same end-to-end trace.


In [11]:
logger = GalileoLogger(project=PROJECT_NAME, log_stream=LOG_STREAM_NAME, mode='distributed')
logger.start_trace(input='Distributed tracing test')
logger.add_llm_span(
    input=[Message(role=MessageRole.user, content='Test')],
    output=Message(role=MessageRole.assistant, content='OK'),
    model=MODEL,
)
headers = logger.get_tracing_headers()
logger.conclude(output='OK')
logger.flush()
headers


{'X-Galileo-Trace-ID': '06001868-f8ec-4ab4-b500-5b0c69b0a496',
 'X-Galileo-Parent-ID': '06001868-f8ec-4ab4-b500-5b0c69b0a496'}

## 8. Clean Up the Demo Project


In [ ]:
delete_project(name=PROJECT_NAME)
